# 🧹 Notebook 02 : Nettoyage des Données

**Auteur** : Oumaro Titans DJIGUIMDE  
**Date** : Février 2026  
**Objectif** : Nettoyer le dataset et préparer des données de qualité pour l'analyse

---

## 🎯 Objectifs de ce notebook

1. Traiter les valeurs manquantes
2. Supprimer les doublons
3. Corriger les valeurs aberrantes
4. Valider les types de données
5. Standardiser les formats
6. Créer un dataset propre pour l'analyse

## 📦 Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Bibliothèques importées")

## 1️⃣ Chargement et état initial

In [ ]:
# Chargement
df_raw = pd.read_csv('../data/sales_data.csv', parse_dates=['date'])

print("="*70)
print("📊 ÉTAT INITIAL DU DATASET")
print("="*70)
print(f"Lignes : {len(df_raw):,}")
print(f"Colonnes : {len(df_raw.columns)}")
print(f"Taille : {df_raw.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print("="*70)

# Copie pour le nettoyage
df = df_raw.copy()

In [ ]:
# Aperçu des problèmes
print("\n⚠️  PROBLÈMES DÉTECTÉS")
print("="*70)
print("\n1. Valeurs manquantes :")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n2. Doublons :")
print(f"   Lignes dupliquées : {df.duplicated().sum()}")

print("\n3. Valeurs aberrantes potentielles :")
print(f"   Prix < 100 FCFA : {len(df[df['prix'] < 100])}")
print(f"   Prix > 1M FCFA : {len(df[df['prix'] > 1000000])}")
print(f"   Quantités <= 0 : {len(df[df['quantite'] <= 0])}")
print(f"   Quantités > 1000 : {len(df[df['quantite'] > 1000])}")
print("="*70)

## 2️⃣ Traitement des doublons

In [ ]:
print("🔄 SUPPRESSION DES DOUBLONS")
print("="*70)

# Avant
nb_avant = len(df)
doublons_avant = df.duplicated().sum()

print(f"Avant : {nb_avant:,} lignes ({doublons_avant} doublons)")

# Suppression des doublons (garde la première occurrence)
df = df.drop_duplicates(keep='first')

# Après
nb_apres = len(df)
doublons_apres = df.duplicated().sum()

print(f"Après : {nb_apres:,} lignes ({doublons_apres} doublons)")
print(f"✅ {nb_avant - nb_apres} lignes supprimées")
print("="*70)

## 3️⃣ Traitement des valeurs manquantes

In [ ]:
print("⚠️  TRAITEMENT DES VALEURS MANQUANTES")
print("="*70)

# État avant
print("\nAvant nettoyage :")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Stratégie 1 : Prix manquants
# On impute avec le prix médian du produit
print("\n📌 Traitement des prix manquants...")

prix_manquants = df['prix'].isnull().sum()
if prix_manquants > 0:
    # Calculer le prix médian par produit
    prix_median_produit = df.groupby('produit')['prix'].median()
    
    # Imputer les valeurs manquantes
    for idx in df[df['prix'].isnull()].index:
        produit = df.loc[idx, 'produit']
        if produit in prix_median_produit.index:
            df.loc[idx, 'prix'] = prix_median_produit[produit]
    
    print(f"✅ {prix_manquants} prix imputés avec la médiane du produit")
else:
    print("✅ Aucun prix manquant")

In [ ]:
# Stratégie 2 : Quantités manquantes
# On impute avec 1 (valeur la plus fréquente)
print("\n📌 Traitement des quantités manquantes...")

qty_manquantes = df['quantite'].isnull().sum()
if qty_manquantes > 0:
    df['quantite'].fillna(1, inplace=True)
    print(f"✅ {qty_manquantes} quantités imputées avec 1")
else:
    print("✅ Aucune quantité manquante")

In [ ]:
# Stratégie 3 : Mode de paiement manquant
# On impute avec le mode le plus fréquent (Orange Money)
print("\n📌 Traitement des modes de paiement manquants...")

pmt_manquants = df['mode_paiement'].isnull().sum()
if pmt_manquants > 0:
    mode_frequent = df['mode_paiement'].mode()[0]
    df['mode_paiement'].fillna(mode_frequent, inplace=True)
    print(f"✅ {pmt_manquants} modes de paiement imputés avec '{mode_frequent}'")
else:
    print("✅ Aucun mode de paiement manquant")

In [ ]:
# Recalcul du chiffre d'affaires après imputation
df['chiffre_affaires'] = df['prix'] * df['quantite']

# Vérification
print("\n✅ Après nettoyage :")
valeurs_manquantes = df.isnull().sum()
if valeurs_manquantes.sum() == 0:
    print("   Aucune valeur manquante ! 🎉")
else:
    print(valeurs_manquantes[valeurs_manquantes > 0])

print("="*70)

## 4️⃣ Correction des valeurs aberrantes

In [ ]:
print("💸 CORRECTION DES PRIX ABERRANTS")
print("="*70)

# Définir les seuils raisonnables par catégorie
seuils_prix = {
    'Alimentation': (500, 100000),
    'Électronique': (1000, 1000000),
    'Textile': (1000, 100000),
    'Électroménager': (5000, 500000),
    'Cosmétiques': (500, 50000),
    'Construction': (100, 100000)
}

nb_corrections = 0

for categorie, (min_prix, max_prix) in seuils_prix.items():
    mask_cat = df['categorie'] == categorie
    
    # Prix trop bas
    mask_bas = mask_cat & (df['prix'] < min_prix)
    if mask_bas.sum() > 0:
        prix_median = df[mask_cat]['prix'].median()
        df.loc[mask_bas, 'prix'] = prix_median
        print(f"{categorie:20s} : {mask_bas.sum():3d} prix trop bas corrigés → {prix_median:,.0f} FCFA")
        nb_corrections += mask_bas.sum()
    
    # Prix trop élevés
    mask_haut = mask_cat & (df['prix'] > max_prix)
    if mask_haut.sum() > 0:
        prix_median = df[mask_cat]['prix'].median()
        df.loc[mask_haut, 'prix'] = prix_median
        print(f"{categorie:20s} : {mask_haut.sum():3d} prix trop élevés corrigés → {prix_median:,.0f} FCFA")
        nb_corrections += mask_haut.sum()

print(f"\n✅ Total : {nb_corrections} prix corrigés")
print("="*70)

In [ ]:
print("\n📊 CORRECTION DES QUANTITÉS ABERRANTES")
print("="*70)

# Quantités négatives ou nulles → 1
mask_qty_neg = df['quantite'] <= 0
nb_qty_neg = mask_qty_neg.sum()
if nb_qty_neg > 0:
    df.loc[mask_qty_neg, 'quantite'] = 1
    print(f"✅ {nb_qty_neg} quantités négatives/nulles corrigées → 1")

# Quantités anormalement élevées (> 1000)
# Vérifier si c'est cohérent avec la catégorie
mask_qty_high = df['quantite'] > 1000
nb_qty_high = mask_qty_high.sum()

if nb_qty_high > 0:
    print(f"\n⚠️  {nb_qty_high} quantités > 1000 détectées :")
    
    for idx in df[mask_qty_high].index:
        categorie = df.loc[idx, 'categorie']
        produit = df.loc[idx, 'produit']
        qty = df.loc[idx, 'quantite']
        
        # Construction : quantités élevées normales (briques, fer)
        if categorie == 'Construction' and qty <= 10000:
            continue  # Normal
        else:
            # Corriger avec la médiane de la catégorie
            qty_median = df[df['categorie'] == categorie]['quantite'].median()
            df.loc[idx, 'quantite'] = qty_median
            print(f"  • {produit} ({categorie}) : {qty} → {qty_median}")

print("\n✅ Quantités corrigées")
print("="*70)

In [ ]:
# Recalcul du chiffre d'affaires après corrections
df['chiffre_affaires'] = df['prix'] * df['quantite']
print("✅ Chiffre d'affaires recalculé")

## 5️⃣ Validation et standardisation

In [ ]:
print("🔍 VALIDATION DES TYPES DE DONNÉES")
print("="*70)

# Vérifier les types
print("\nTypes actuels :")
print(df.dtypes)

# Conversion si nécessaire
df['prix'] = df['prix'].astype(float)
df['quantite'] = df['quantite'].astype(float)
df['chiffre_affaires'] = df['chiffre_affaires'].astype(float)

print("\n✅ Types validés et standardisés")
print("="*70)

In [ ]:
print("\n📝 STANDARDISATION DES FORMATS")
print("="*70)

# Supprimer les espaces superflus
colonnes_str = ['produit', 'categorie', 'ville', 'mode_paiement', 'vendeur']

for col in colonnes_str:
    df[col] = df[col].str.strip()

print("✅ Espaces superflus supprimés")

# Vérifier la cohérence des valeurs catégorielles
print("\n📊 Valeurs uniques par colonne :")
print(f"  • Villes : {df['ville'].nunique()}")
print(f"  • Catégories : {df['categorie'].nunique()}")
print(f"  • Modes paiement : {df['mode_paiement'].nunique()}")
print(f"  • Produits : {df['produit'].nunique()}")
print(f"  • Vendeurs : {df['vendeur'].nunique()}")

print("="*70)

## 6️⃣ Ajout de colonnes dérivées

In [ ]:
print("➕ AJOUT DE COLONNES DÉRIVÉES")
print("="*70)

# Extraction de composantes temporelles
df['annee'] = df['date'].dt.year
df['mois'] = df['date'].dt.month
df['mois_nom'] = df['date'].dt.month_name()
df['jour'] = df['date'].dt.day
df['jour_semaine'] = df['date'].dt.day_name()
df['trimestre'] = df['date'].dt.quarter
df['semaine'] = df['date'].dt.isocalendar().week

print("✅ Colonnes temporelles ajoutées :")
print("   • annee, mois, mois_nom, jour")
print("   • jour_semaine, trimestre, semaine")

# Catégories de prix
df['categorie_prix'] = pd.cut(df['prix'], 
                              bins=[0, 10000, 50000, 100000, float('inf')],
                              labels=['Bas', 'Moyen', 'Élevé', 'Premium'])

print("\n✅ Catégorie de prix ajoutée (Bas, Moyen, Élevé, Premium)")

# Flag weekend
df['est_weekend'] = df['date'].dt.dayofweek.isin([5, 6])
print("✅ Flag weekend ajouté")

print("="*70)

## 7️⃣ Rapport de nettoyage

In [ ]:
print("="*70)
print("📋 RAPPORT DE NETTOYAGE")
print("="*70)

print(f"\n📊 Dataset initial :")
print(f"   • Lignes : {len(df_raw):,}")
print(f"   • Colonnes : {len(df_raw.columns)}")

print(f"\n📊 Dataset nettoyé :")
print(f"   • Lignes : {len(df):,}")
print(f"   • Colonnes : {len(df.columns)}")
print(f"   • Lignes supprimées : {len(df_raw) - len(df):,}")

print(f"\n✅ Corrections effectuées :")
print(f"   • Doublons supprimés : {df_raw.duplicated().sum()}")
print(f"   • Valeurs manquantes traitées")
print(f"   • Prix aberrants corrigés")
print(f"   • Quantités aberrantes corrigées")
print(f"   • Colonnes dérivées ajoutées : {len(df.columns) - len(df_raw.columns)}")

print(f"\n💰 Statistiques du dataset nettoyé :")
print(f"   • CA total : {df['chiffre_affaires'].sum():,.0f} FCFA")
print(f"   • CA moyen : {df['chiffre_affaires'].mean():,.0f} FCFA")
print(f"   • CA médian : {df['chiffre_affaires'].median():,.0f} FCFA")

print(f"\n🎯 Qualité des données :")
print(f"   • Valeurs manquantes : {df.isnull().sum().sum()} (0%)")
print(f"   • Doublons : {df.duplicated().sum()} (0%)")
print(f"   • Lignes complètes : {len(df):,} (100%)")

print("\n" + "="*70)

## 8️⃣ Visualisation avant/après

In [ ]:
# Comparaison distribution des prix
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Avant
axes[0].hist(df_raw['prix'].dropna(), bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution des Prix - AVANT nettoyage', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Prix (FCFA)')
axes[0].set_ylabel('Fréquence')
axes[0].grid(True, alpha=0.3)

# Après
axes[1].hist(df['prix'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution des Prix - APRÈS nettoyage', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Prix (FCFA)')
axes[1].set_ylabel('Fréquence')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparaison CA
print("💰 COMPARAISON DU CHIFFRE D'AFFAIRES")
print("="*70)
print(f"CA avant nettoyage : {df_raw['chiffre_affaires'].sum():,.0f} FCFA")
print(f"CA après nettoyage : {df['chiffre_affaires'].sum():,.0f} FCFA")
diff = df['chiffre_affaires'].sum() - df_raw['chiffre_affaires'].sum()
print(f"Différence : {diff:,.0f} FCFA ({(diff/df_raw['chiffre_affaires'].sum()*100):.2f}%)")
print("="*70)

## 9️⃣ Sauvegarde du dataset nettoyé

In [ ]:
# Sauvegarde
output_file = '../data/sales_data_clean.csv'
df.to_csv(output_file, index=False, encoding='utf-8')

print("="*70)
print("💾 SAUVEGARDE DU DATASET NETTOYÉ")
print("="*70)
print(f"Fichier : {output_file}")
print(f"Lignes : {len(df):,}")
print(f"Colonnes : {len(df.columns)}")
print(f"Taille : {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print("\n✅ Dataset nettoyé et prêt pour l'analyse SQL !")
print("="*70)

In [ ]:
# Aperçu final
print("\n👀 Aperçu du dataset nettoyé :")
df.head(10)

In [ ]:
# Statistiques finales
print("\n📊 Statistiques descriptives finales :")
df[['prix', 'quantite', 'chiffre_affaires']].describe()

## 📌 Résumé et prochaines étapes

### ✅ Ce qui a été fait

1. **Doublons** : Suppression de ~228 lignes dupliquées
2. **Valeurs manquantes** : Imputation intelligente
   - Prix : médiane par produit
   - Quantités : valeur 1
   - Mode paiement : mode le plus fréquent
3. **Valeurs aberrantes** : Correction avec médianes par catégorie
4. **Enrichissement** : Ajout de 8 colonnes dérivées
5. **Validation** : Vérification de la cohérence

### 🎯 Qualité du dataset final

- ✅ 0% de valeurs manquantes
- ✅ 0% de doublons
- ✅ Valeurs dans des plages cohérentes
- ✅ Types de données validés
- ✅ Colonnes temporelles enrichies

### 🚀 Prochaine étape

**Notebook 03** : Analyses SQL avancées
- Requêtes complexes avec JOINs
- Window functions (ROW_NUMBER, RANK, LAG, LEAD)
- Analyses de cohortes
- Calcul de KPIs business
- Analyses prédictives

Le dataset est maintenant **prêt pour l'analyse SQL** ! 🎉